# 🏦 Programa de Especialización en Credit Scoring con Python
## Sesión 9 — Credit Scoring Tradicional vs Machine Learning

---

**Curso II: Credit Scoring y Machine Learning**  
**Docente:** Enzo Infantes Zúñiga — MSc Data Science, Barcelona School of Economics  
**Repositorio:** https://github.com/Enzo280100/CreditScoring  

---

> *"El enfoque tradicional del scoring crediticio nos enseñó la disciplina de la interpretabilidad y la regulación. El Machine Learning nos ofrece el poder predictivo. La pregunta no es cuál usar, sino cuándo, cómo y por qué."*

---

## 📋 Agenda de la Sesión

| # | Bloque | Descripción |
|---|--------|-------------|
| 1 | **Contexto y Motivación** | Recapitulación del Curso I y por qué transicionar a ML |
| 2 | **Panorama del Ecosistema ML** | Taxonomía de modelos y su posición en el ciclo de crédito |
| 3 | **Regresión Logística — Revisión Crítica** | Supuestos, fortalezas y límites reales |
| 4 | **Comparativa Conceptual** | Tradicional vs ML en 7 dimensiones clave |
| 5 | **Benchmarking Empírico** | Comparación numérica sobre dataset real |
| 6 | **Framework de Decisión** | ¿Cuándo usar cada enfoque? |
| 7 | **Gobernanza y Regulación** | SR 11-7, EBA, Basilea — implicancias prácticas |
| 8 | **Conclusiones y Roadmap** | Qué viene en las sesiones 10–18 |

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el participante será capaz de:

1. Reconocer las **limitaciones técnicas y regulatorias** de los modelos tradicionales de scoring.
2. Identificar y clasificar los principales **algoritmos de Machine Learning** aplicables al crédito.
3. Comparar ambos enfoques en dimensiones de **rendimiento, interpretabilidad y gobernanza**.
4. Ejecutar un **benchmark empírico** sobre un dataset de crédito real.
5. Articular un **framework de decisión** para seleccionar el enfoque adecuado según el contexto institucional.

---

# 📦 Configuración del Entorno

In [ ]:
# ============================================================
#  SESIÓN 9 — CREDIT SCORING TRADICIONAL VS MACHINE LEARNING
#  Programa de Especialización en Credit Scoring con Python
# ============================================================

# --- Librerías estándar ---
import warnings
import time
warnings.filterwarnings('ignore')

# --- Manipulación de datos ---
import numpy as np
import pandas as pd

# --- Visualización ---
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# --- Scikit-learn: preprocesamiento y pipelines ---
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# --- Scikit-learn: modelos ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# --- Scikit-learn: métricas ---
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

# --- Scikit-learn: datasets demo ---
from sklearn.datasets import make_classification

# --- Estilo global de gráficos ---
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2C5F8A', '#E8523A', '#2CA02C', '#FF7F0E', '#9467BD', '#8C564B']
sns.set_palette(PALETTE)

print("✅ Entorno configurado correctamente.")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")
print(f"   Sklearn: {__import__('sklearn').__version__}")

---

# 🔁 Sección 1 — Recapitulación y Motivación

## 1.1 ¿Qué construimos en el Curso I?

En las primeras ocho sesiones construimos el **pipeline clásico de credit scoring** basado en regresión logística:

```
Datos Brutos
    ↓
Definición del Evento de Incumplimiento (Sesión 2)
    ↓
EDA + Tratamiento de variables (Sesión 3)
    ↓
WOE / IV (Sesión 4)
    ↓
Regresión Logística (Sesión 5)
    ↓
Evaluación: Gini, KS, AUC (Sesión 6)
    ↓
Validación + PSI/CSI (Sesión 7)
    ↓
Scorecard + Puntos de corte (Sesión 8)
```

Este pipeline es **sólido, regulatoriamente aceptado y ampliamente utilizado** en la banca tradicional. Pero tiene limitaciones que exploraremos en esta sesión.

## 1.2 ¿Por qué transicionar hacia Machine Learning?

Cuatro fuerzas impulsoras del cambio en la industria financiera:

| Fuerza | Descripción |
|--------|-------------|
| 📊 **Volumen y variedad de datos** | Datos alternativos (transaccionales, de comportamiento, open banking) que la regresión logística no aprovecha bien |
| ⚡ **Poder computacional** | La nube y los GPUs hacen entrenable en minutos modelos que antes tardaban días |
| 🎯 **Presión por performance** | Mercados competitivos exigen discriminación predictiva más fina |
| 🏛️ **Evolución regulatoria** | IFRS 9, Basilea IV y las guidelines de la EBA permiten (con condiciones) modelos no lineales |

> **Advertencia temprana:** ML no es sinónimo de "mejor". En muchos contextos bancarios regulados, una regresión logística bien calibrada supera a un modelo complejo mal gobernado.

In [ ]:
# =============================================================================
# VISUALIZACIÓN: Evolución del AUC típico por tipo de modelo y tipo de datos
# (Basado en benchmarks de la industria — valores ilustrativos)
# =============================================================================

modelos = ['Scorecard\nManual', 'Reg. Logística\n+ WOE', 'Decision\nTree', 
           'Random\nForest', 'Gradient\nBoosting', 'Deep\nLearning']

auc_datos_clasicos    = [0.68, 0.72, 0.70, 0.76, 0.78, 0.77]   # solo variables bureau
auc_datos_alternativos= [0.66, 0.71, 0.72, 0.80, 0.84, 0.86]   # variables transaccionales incluidas

x = np.arange(len(modelos))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, auc_datos_clasicos,    width, label='Datos Clásicos (Bureau)',       color=PALETTE[0], alpha=0.85, edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, auc_datos_alternativos,width, label='Datos Alternativos (Transacc.)',color=PALETTE[1], alpha=0.85, edgecolor='white', linewidth=0.8)

ax.set_xlabel('Tipo de Modelo', fontsize=12)
ax.set_ylabel('AUC-ROC Típico', fontsize=12)
ax.set_title('Ganancia de Discriminación: Modelos Tradicionales vs ML\n(Benchmark Ilustrativo — Portafolios de Consumo)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(modelos, fontsize=10)
ax.set_ylim(0.60, 0.92)
ax.axhline(0.70, color='gray', linestyle='--', alpha=0.5, label='Umbral mínimo típico (0.70)')
ax.axhline(0.80, color='green', linestyle='--', alpha=0.5, label='Umbral "bueno" (0.80)')
ax.legend(fontsize=10)

# Etiquetas de valor
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color=PALETTE[0], fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color=PALETTE[1], fontweight='bold')

# Zona sombreada: zona de los modelos clásicos
ax.axvspan(-0.5, 1.5, alpha=0.05, color='blue', label='Zona Modelos Clásicos')
ax.axvspan(1.5, 5.5, alpha=0.05, color='red',  label='Zona ML')
ax.text(0.5, 0.87, 'Zona\nClásica', ha='center', fontsize=9, color='navy', alpha=0.7)
ax.text(3.5, 0.87, 'Zona Machine Learning', ha='center', fontsize=9, color='darkred', alpha=0.7)

plt.tight_layout()
plt.savefig('../figuras/sesion09_benchmark_auc_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: figuras/sesion09_benchmark_auc_overview.png")

---

# 🗺️ Sección 2 — Panorama del Ecosistema ML en Credit Scoring

## 2.1 Taxonomía de Algoritmos para Clasificación Binaria

Los modelos que usaremos en el Curso II se organizan en tres familias:

### Familia 1: Modelos Basados en Árboles
| Algoritmo | Tipo | Sesión | Complejidad |
|-----------|------|--------|-------------|
| Decision Tree | Árbol único | 11 | Baja |
| Random Forest | Ensemble bagging | 11 | Media |
| Gradient Boosting | Ensemble boosting | 12 | Alta |
| XGBoost | Boosting optimizado | 12 | Alta |
| LightGBM | Boosting ultra-eficiente | 12 | Alta |

### Familia 2: Modelos Lineales Extendidos
| Algoritmo | Tipo | Uso en Credit |
|-----------|------|---------------|
| Regresión Logística | Lineal | Baseline + scorecard |
| Logística Regularizada (L1/L2) | Lineal + penalización | Selección de variables |
| Elastic Net | Lineal + penalización mixta | Alta dimensionalidad |

### Familia 3: Modelos No Paramétricos Avanzados
| Algoritmo | Característica | Limitación en Credit |
|-----------|----------------|----------------------|
| SVM | Buen discriminador en fronteras complejas | Difícil interpretación |
| KNN | Simple y local | No escala bien |
| Redes Neuronales | Altísima capacidad | Caja negra, regulatorio complejo |

## 2.2 Posición en el Ciclo de Crédito

Los modelos de scoring se usan en distintos momentos del ciclo crediticio, y el enfoque óptimo varía:

| Etapa | Modelo Típico | ¿ML Agrega Valor? |
|-------|--------------|-------------------|
| **Originación** (application scoring) | Logística + WOE | Alto, especialmente con datos alternativos |
| **Comportamiento** (behavioral scoring) | Logística o RF | Muy alto (datos transaccionales) |
| **Cobranza** (collection scoring) | Logística | Medio-alto |
| **Fraude** | Ensemble / NN | Muy alto |
| **IFRS 9 PD** | Logística / RF | Medio (requiere interpretabilidad) |

In [ ]:
# =============================================================================
# VISUALIZACIÓN: Mapa de modelos — Interpretabilidad vs Capacidad Predictiva
# =============================================================================

modelos_map = {
    'Scorecard\nManual':       {'interpretabilidad': 9.5, 'capacidad': 5.5, 'uso': 8.0, 'color': PALETTE[0]},
    'Reg. Logística':          {'interpretabilidad': 8.5, 'capacidad': 6.5, 'uso': 9.0, 'color': PALETTE[0]},
    'Logística\nRegularizada': {'interpretabilidad': 7.5, 'capacidad': 7.0, 'uso': 7.0, 'color': PALETTE[0]},
    'Decision\nTree':          {'interpretabilidad': 7.0, 'capacidad': 6.0, 'uso': 5.0, 'color': PALETTE[2]},
    'Random\nForest':          {'interpretabilidad': 4.5, 'capacidad': 8.0, 'uso': 7.5, 'color': PALETTE[2]},
    'XGBoost /\nLightGBM':     {'interpretabilidad': 3.5, 'capacidad': 9.0, 'uso': 8.5, 'color': PALETTE[1]},
    'SVM':                     {'interpretabilidad': 2.5, 'capacidad': 7.5, 'uso': 3.0, 'color': PALETTE[4]},
    'Red Neuronal':            {'interpretabilidad': 1.5, 'capacidad': 9.5, 'uso': 4.0, 'color': PALETTE[4]},
}

fig, ax = plt.subplots(figsize=(11, 7))

for nombre, vals in modelos_map.items():
    ax.scatter(vals['interpretabilidad'], vals['capacidad'],
               s=vals['uso'] * 60, color=vals['color'], alpha=0.8,
               edgecolors='white', linewidth=1.5, zorder=5)
    ax.annotate(nombre,
                (vals['interpretabilidad'], vals['capacidad']),
                textcoords='offset points', xytext=(8, 5),
                fontsize=9, ha='left')

# Zonas
ax.axvspan(6.5, 10.5, alpha=0.06, color='blue')
ax.axvspan(0.5, 6.5, alpha=0.06, color='red')
ax.text(8.5, 9.5, 'Zona Regulatoria\n"Caja Blanca"', ha='center', fontsize=10, color='navy', alpha=0.6, fontweight='bold')
ax.text(3.5, 9.5, 'Zona ML Avanzada\n"Caja Negra"', ha='center', fontsize=10, color='darkred', alpha=0.6, fontweight='bold')

ax.set_xlabel('← Menor Interpretabilidad   |   Mayor Interpretabilidad →', fontsize=11)
ax.set_ylabel('← Menor Capacidad Predictiva   |   Mayor Capacidad →', fontsize=11)
ax.set_title('Mapa de Modelos de Credit Scoring\nInterpretabilidad vs Capacidad Predictiva\n(Tamaño = Frecuencia de uso en banca comercial)', 
             fontsize=12, fontweight='bold')
ax.set_xlim(0.5, 10.5)
ax.set_ylim(4.5, 10.5)
ax.axvline(6.5, color='gray', linestyle='--', alpha=0.4)
ax.invert_xaxis()

# Leyenda custom
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=PALETTE[0], markersize=12, label='Modelos Tradicionales'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=PALETTE[2], markersize=12, label='Árboles / Ensembles'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=PALETTE[1], markersize=12, label='Boosting'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=PALETTE[4], markersize=12, label='Otros ML'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../figuras/sesion09_mapa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 📐 Sección 3 — Regresión Logística: Revisión Crítica

## 3.1 El Modelo que Domina la Industria

La regresión logística es, con diferencia, el modelo más utilizado en credit scoring institucional. Para entender por qué ML puede agregarle valor, primero debemos entender **profundamente** sus supuestos y limitaciones.

### Formulación del Modelo

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p)}}$$

donde $Y=1$ indica incumplimiento (default).

### Supuestos Críticos

| Supuesto | Descripción | ¿Se cumple en crédito? |
|----------|-------------|------------------------|
| **Linealidad en log-odds** | La relación $X_i \rightarrow \text{logit}(P)$ es lineal | Frecuentemente **no** (por eso usamos WOE) |
| **Independencia de observaciones** | Las observaciones son i.i.d. | Parcialmente (correlación temporal) |
| **Ausencia de multicolinealidad** | Las variables predictoras no están correladas | Problemático con bureau data |
| **Sin valores extremos influyentes** | Outliers no distorsionan los parámetros | Relevante en variables monetarias |
| **Muestra suficiente** | Regla heurística: ≥10 eventos por variable | A veces violado en niche products |

## 3.2 Fortalezas que ML No Siempre Supera

```
✅ Coeficientes directamente interpretables como odds ratios
✅ Produce probabilidades calibradas de forma nativa
✅ Robusto ante overfitting si se construye con parsimonia
✅ Fácil de auditar y defender ante reguladores
✅ Estabilidad: el modelo "envejece" de forma predecible
✅ Bajo costo computacional de inferencia (producción)
```

## 3.3 Limitaciones Reales

```
❌ No captura interacciones automáticamente
❌ No maneja bien relaciones no lineales (sin transformaciones manuales)
❌ Sensible a la separación perfecta (convergencia fallida)
❌ Desaprovecha datos de alta dimensionalidad (hundreds of features)
❌ Requiere ingeniería de variables intensiva (WOE, binning manual)
❌ Menor capacidad discriminante en presencia de patrones complejos
```

In [ ]:
# =============================================================================
# DEMOSTRACIÓN: Limitación de la regresión logística con relaciones no lineales
# Compararemos RL vs Decision Tree en un problema sintético con interacciones
# =============================================================================

np.random.seed(42)
n = 2000

# Variables sintéticas con estructura no lineal
# (como ocurre con edad-ingresos-deuda en credit scoring)
edad     = np.random.uniform(18, 70, n)
ingresos = np.random.exponential(3000, n) + 500
deuda    = np.random.uniform(0, 1, n)  # ratio deuda/ingreso

# Probabilidad de default: parabólica en edad, umbral en deuda
log_odds = (
    -0.05 * (edad - 40)**2 / 100   # jóvenes y viejos tienen más riesgo
    + 0.3  * (deuda > 0.6).astype(float)  # interacción no lineal
    - 0.0001 * ingresos
    + 0.5
)
prob_default = 1 / (1 + np.exp(-log_odds))
y_sintetico  = (np.random.uniform(0, 1, n) < prob_default).astype(int)

X_sint = pd.DataFrame({'edad': edad, 'ingresos': ingresos, 'ratio_deuda': deuda})

X_tr, X_te, y_tr, y_te = train_test_split(X_sint, y_sintetico, test_size=0.3, random_state=42, stratify=y_sintetico)

# Regresión Logística
rl = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
rl.fit(X_tr, y_tr)
prob_rl = rl.predict_proba(X_te)[:,1]
auc_rl  = roc_auc_score(y_te, prob_rl)

# Árbol de Decisión
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_tr, y_tr)
prob_dt = dt.predict_proba(X_te)[:,1]
auc_dt  = roc_auc_score(y_te, prob_dt)

# Visualización comparativa ROC
fpr_rl, tpr_rl, _ = roc_curve(y_te, prob_rl)
fpr_dt, tpr_dt, _ = roc_curve(y_te, prob_dt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
ax = axes[0]
ax.plot(fpr_rl, tpr_rl, color=PALETTE[0], lw=2, label=f'Reg. Logística  (AUC = {auc_rl:.4f})')
ax.plot(fpr_dt, tpr_dt, color=PALETTE[1], lw=2, label=f'Árbol Decisión  (AUC = {auc_dt:.4f})')
ax.plot([0,1],[0,1],'k--', alpha=0.5, lw=1)
ax.fill_between(fpr_rl, tpr_rl, alpha=0.1, color=PALETTE[0])
ax.fill_between(fpr_dt, tpr_dt, alpha=0.1, color=PALETTE[1])
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
ax.set_title('Curva ROC\nProblema Sintético con No Linealidades', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_aspect('equal')

# Distribución de scores por clase
ax = axes[1]
ax.hist(prob_rl[y_te==0], bins=40, alpha=0.5, color=PALETTE[0], label='No Default (RL)', density=True)
ax.hist(prob_rl[y_te==1], bins=40, alpha=0.5, color=PALETTE[0], histtype='step', linewidth=2, label='Default (RL)', density=True)
ax.hist(prob_dt[y_te==0], bins=40, alpha=0.5, color=PALETTE[1], label='No Default (DT)', density=True)
ax.hist(prob_dt[y_te==1], bins=40, alpha=0.5, color=PALETTE[1], histtype='step', linewidth=2, label='Default (DT)', density=True)
ax.set_xlabel('Probabilidad Predicha de Default', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title('Distribución de Scores\npor Clase Real', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figuras/sesion09_limitaciones_rl.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Resultados en Datos Sintéticos con No Linealidades:")
print(f"   Regresión Logística  → AUC = {auc_rl:.4f}")
print(f"   Árbol de Decisión    → AUC = {auc_dt:.4f}")
print(f"   Ganancia ML          → +{(auc_dt - auc_rl)*100:.2f} puntos AUC")
print()
print("💡 Observación: El árbol captura mejor las no linealidades sintéticas.")
print("   En datos reales de banca, esta diferencia suele ser menor (2-5 puntos Gini).")

---

# ⚖️ Sección 4 — Comparativa Conceptual: 7 Dimensiones Clave

## 4.1 Las Siete Dimensiones de Evaluación

Para una comparación justa y útil para la banca, debemos evaluar los modelos en siete dimensiones que los comités de riesgo y los reguladores consideran relevantes:

```
1. Poder Discriminante        → ¿Qué tan bien separa buenos de malos?
2. Calibración                → ¿Las probabilidades predichas son confiables?
3. Interpretabilidad          → ¿Podemos explicar cada predicción?
4. Estabilidad                → ¿El modelo se degrada lentamente?
5. Requerimientos de Datos    → ¿Cuántos datos necesita para funcionar bien?
6. Costo de Implementación    → ¿Qué tan fácil es poner en producción?
7. Cumplimiento Regulatorio   → ¿Lo acepta el regulador?
```

## 4.2 Tabla Comparativa Detallada

| Dimensión | Regresión Logística | Random Forest | Gradient Boosting |
|-----------|--------------------|--------------|-----------------|
| **Discriminación** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Calibración nativa** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **Interpretabilidad global** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Interpretabilidad local** | ⭐⭐⭐⭐⭐ | ⭐⭐ (con SHAP) | ⭐⭐ (con SHAP) |
| **Estabilidad temporal** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Muestra pequeña** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Alta dimensionalidad** | ⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Facilidad de auditoría** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Aceptación regulatoria** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |

*Escala: ⭐ = Deficiente, ⭐⭐⭐⭐⭐ = Excelente*

## 4.3 El Dilema Interpretabilidad-Performance

Este es el **trade-off central** de la sesión:

> **A mayor complejidad del modelo → Mayor capacidad de capturar patrones → Menor interpretabilidad directa**

La buena noticia: las herramientas de **Explainable AI (XAI)** — que veremos en la Sesión 14 — están reduciendo este trade-off con técnicas como SHAP, LIME y PDP.

In [ ]:
# =============================================================================
# VISUALIZACIÓN: Radar Chart de las 7 dimensiones por modelo
# =============================================================================

from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

categorias = ['Discriminación', 'Calibración\nNativa', 'Interpretab.\nGlobal',
              'Estabilidad', 'Muestra\nPequeña', 'Alta\nDimension.', 'Aceptación\nReg.']

scores = {
    'Reg. Logística':   [3, 5, 5, 4, 5, 2, 5],
    'Random Forest':    [4, 3, 3, 4, 3, 4, 4],
    'Gradient Boost.':  [5, 3, 2, 3, 2, 5, 3],
}

N = len(categorias)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # cerrar el polígono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colores = [PALETTE[0], PALETTE[2], PALETTE[1]]

for (nombre, vals), color in zip(scores.items(), colores):
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, '-o', color=color, linewidth=2.5, label=nombre)
    ax.fill(angles, vals_plot, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categorias, fontsize=11, fontweight='bold')
ax.set_ylim(0, 5.5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=8, color='gray')
ax.set_title('Comparativa en 7 Dimensiones\nModelos de Credit Scoring', 
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.savefig('../figuras/sesion09_radar_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 🔬 Sección 5 — Benchmarking Empírico sobre Dataset Real

## 5.1 Descripción del Dataset

Utilizamos el dataset **German Credit** (también conocido como *Statlog German Credit Data*), uno de los más usados en la literatura de credit scoring:

- **Origen:** UCI Machine Learning Repository / Hofmann (1994)
- **Registros:** 1,000 clientes
- **Variables:** 20 predictores (mix cuantitativo y categórico)
- **Target:** 1 = Crédito riesgoso, 0 = Crédito bueno
- **Tasa de default:** 30%

Si en tu entorno de trabajo dispones del dataset del Curso I, reemplaza la carga de datos con:
```python
df = pd.read_csv('../data/credit_data.csv')
```

In [ ]:
# =============================================================================
# CARGA Y PREPARACIÓN DE DATOS — German Credit Dataset
# =============================================================================

# Intentar cargar desde el repositorio local primero
import os

DATA_PATH = '../data/german_credit.csv'

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset cargado desde: {DATA_PATH}")
else:
    # Generamos un dataset sintético con estructura similar al German Credit
    print("⚠️  Archivo no encontrado en ruta local.")
    print("   Generando dataset sintético con estructura equivalente...")
    
    np.random.seed(2024)
    n_obs = 1000
    
    df = pd.DataFrame({
        # Variables numéricas
        'duracion_meses':      np.random.choice([6, 12, 18, 24, 36, 48, 60, 72], n_obs),
        'monto_credito':       np.random.lognormal(7.5, 0.8, n_obs).astype(int),
        'tasa_plazo':          np.random.choice([1, 2, 3, 4], n_obs),
        'edad':                np.random.randint(18, 75, n_obs),
        'num_creditos':        np.random.choice([1, 2, 3, 4], n_obs, p=[0.5, 0.3, 0.15, 0.05]),
        'num_dependientes':    np.random.choice([1, 2], n_obs, p=[0.7, 0.3]),
        'residente_desde':     np.random.choice([1, 2, 3, 4], n_obs),
        
        # Variables categóricas (codificadas numéricamente)
        'estado_cuenta':       np.random.choice([0, 1, 2, 3], n_obs, p=[0.27, 0.27, 0.06, 0.40]),
        'historial_credito':   np.random.choice([0, 1, 2, 3, 4], n_obs, p=[0.04, 0.05, 0.53, 0.18, 0.20]),
        'proposito':           np.random.choice(range(10), n_obs),
        'cuenta_ahorro':       np.random.choice([0, 1, 2, 3, 4], n_obs, p=[0.06, 0.10, 0.10, 0.06, 0.68]),
        'empleo_actual':       np.random.choice([0, 1, 2, 3, 4], n_obs, p=[0.06, 0.17, 0.34, 0.24, 0.20]),
        'estado_personal':     np.random.choice([0, 1, 2, 3], n_obs, p=[0.09, 0.45, 0.31, 0.15]),
        'otros_deudores':      np.random.choice([0, 1, 2], n_obs, p=[0.07, 0.04, 0.89]),
        'propiedad':           np.random.choice([0, 1, 2, 3], n_obs, p=[0.28, 0.10, 0.33, 0.29]),
        'planes_pago':         np.random.choice([0, 1, 2], n_obs, p=[0.14, 0.06, 0.81]),
        'vivienda':            np.random.choice([0, 1, 2], n_obs, p=[0.02, 0.18, 0.71]),
        'trabajo':             np.random.choice([0, 1, 2, 3], n_obs, p=[0.02, 0.20, 0.29, 0.49]),
        'telefono':            np.random.choice([0, 1], n_obs, p=[0.40, 0.60]),
        'extranjero':          np.random.choice([0, 1], n_obs, p=[0.96, 0.04]),
    })
    
    # Target con lógica crediticia realista
    log_odds_gc = (
        -1.0
        + 0.3  * (df['estado_cuenta'] == 0)
        - 0.4  * (df['historial_credito'] >= 3)
        + 0.02 * (df['duracion_meses'] / 12)
        - 0.015* df['edad']
        + 0.5  * (df['cuenta_ahorro'] == 0)
        + 0.3  * (df['empleo_actual'] == 0)
        + np.random.normal(0, 0.5, n_obs)
    )
    prob_gc = 1 / (1 + np.exp(-log_odds_gc))
    df['default'] = (np.random.uniform(0, 1, n_obs) < prob_gc).astype(int)
    
    print(f"✅ Dataset sintético generado: {df.shape[0]} observaciones, {df.shape[1]} variables.")

# Reporte básico
target_col = 'default'
print(f"\n📊 Descripción del Dataset:")
print(f"   Filas       : {df.shape[0]:,}")
print(f"   Columnas    : {df.shape[1]}")
print(f"   Valores NA  : {df.isnull().sum().sum()}")
print(f"   Tasa default: {df[target_col].mean():.1%}")
print(f"   Proporción  : {(df[target_col]==0).sum()} buenos : {(df[target_col]==1).sum()} malos")

In [ ]:
# =============================================================================
# PREPARACIÓN DEL BENCHMARK
# =============================================================================

# Separar features y target
feature_cols = [c for c in df.columns if c != target_col]
X = df[feature_cols]
y = df[target_col]

# División train/test estratificada (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"📦 Partición del dataset:")
print(f"   Train: {X_train.shape[0]:,} obs. | Default rate: {y_train.mean():.1%}")
print(f"   Test : {X_test.shape[0]:,}  obs. | Default rate: {y_test.mean():.1%}")

# Pipeline base: imputación + escalado
# (En las sesiones posteriores veremos feature engineering específico para ML)
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print("\n✅ Preprocesamiento completado (imputación mediana + estandarización).")

In [ ]:
# =============================================================================
# BENCHMARK MULTI-MODELO
# 7 modelos comparados en: AUC-ROC, Gini, KS, Brier Score, Tiempo de entrenamiento
# =============================================================================

modelos_bench = {
    'Reg. Logística':       LogisticRegression(max_iter=1000, random_state=42),
    'Reg. Log. L1 (LASSO)': LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=42),
    'Reg. Log. L2 (Ridge)': LogisticRegression(penalty='l2', max_iter=1000, C=0.5, random_state=42),
    'Decision Tree':        DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42),
    'AdaBoost':             AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
}

resultados = []

print("🔄 Entrenando modelos...\n")
print(f"{'Modelo':<25} {'AUC-ROC':>8} {'Gini':>8} {'KS':>8} {'Brier':>8} {'Tiempo(s)':>10}")
print("-" * 70)

for nombre, modelo in modelos_bench.items():
    t0 = time.time()
    modelo.fit(X_train_proc, y_train)
    tiempo = time.time() - t0
    
    probs = modelo.predict_proba(X_test_proc)[:, 1]
    
    auc    = roc_auc_score(y_test, probs)
    gini   = 2 * auc - 1
    brier  = brier_score_loss(y_test, probs)
    
    # Estadístico KS
    fpr, tpr, _ = roc_curve(y_test, probs)
    ks = np.max(tpr - fpr)
    
    # Validación cruzada AUC (5 folds)
    cv_auc = cross_val_score(
        modelo, X_train_proc, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    ).mean()
    
    resultados.append({
        'Modelo': nombre,
        'AUC-ROC': auc,
        'Gini': gini,
        'KS': ks,
        'Brier Score': brier,
        'CV AUC (5-fold)': cv_auc,
        'Tiempo (s)': tiempo,
        'probs': probs
    })
    
    print(f"{nombre:<25} {auc:>8.4f} {gini:>8.4f} {ks:>8.4f} {brier:>8.4f} {tiempo:>10.3f}")

df_resultados = pd.DataFrame(resultados).drop(columns='probs')
print("\n✅ Benchmark completado.")

In [ ]:
# =============================================================================
# VISUALIZACIÓN COMPLETA DEL BENCHMARK
# =============================================================================

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

nombres = df_resultados['Modelo'].values
colors_bar = [PALETTE[0]]*3 + [PALETTE[2]] + [PALETTE[2]] + [PALETTE[1]]*2

# --- Panel 1: AUC-ROC ---
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.barh(nombres, df_resultados['AUC-ROC'], color=colors_bar, edgecolor='white', linewidth=0.8)
ax1.set_xlabel('AUC-ROC', fontsize=11)
ax1.set_title('AUC-ROC', fontsize=12, fontweight='bold')
ax1.axvline(0.70, color='gray', linestyle='--', alpha=0.6)
for bar, v in zip(bars, df_resultados['AUC-ROC']):
    ax1.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
ax1.set_xlim(0.55, 0.90)

# --- Panel 2: Gini ---
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.barh(nombres, df_resultados['Gini'] * 100, color=colors_bar, edgecolor='white', linewidth=0.8)
ax2.set_xlabel('Coef. Gini (%)', fontsize=11)
ax2.set_title('Coeficiente Gini', fontsize=12, fontweight='bold')
ax2.axvline(40, color='gray', linestyle='--', alpha=0.6)
for bar, v in zip(bars, df_resultados['Gini'] * 100):
    ax2.text(v + 0.5, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
ax2.set_xlim(10, 80)

# --- Panel 3: KS ---
ax3 = fig.add_subplot(gs[0, 2])
bars = ax3.barh(nombres, df_resultados['KS'] * 100, color=colors_bar, edgecolor='white', linewidth=0.8)
ax3.set_xlabel('KS (%)', fontsize=11)
ax3.set_title('Estadístico KS', fontsize=12, fontweight='bold')
ax3.axvline(30, color='gray', linestyle='--', alpha=0.6)
for bar, v in zip(bars, df_resultados['KS'] * 100):
    ax3.text(v + 0.3, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
ax3.set_xlim(10, 75)

# --- Panel 4: ROC Curves superpuestas ---
ax4 = fig.add_subplot(gs[1, 0:2])
for i, (nombre, modelo) in enumerate(modelos_bench.items()):
    probs = resultados[i]['probs']
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = resultados[i]['AUC-ROC']
    color = colors_bar[i]
    lw = 2.5 if 'Gradient' in nombre or 'Forest' in nombre else 1.5
    ls = '-' if 'Gradient' in nombre or 'Forest' in nombre or 'Logística' == nombre else '--'
    ax4.plot(fpr, tpr, color=color, lw=lw, linestyle=ls, alpha=0.85, label=f'{nombre} (AUC={auc:.3f})')
ax4.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4)
ax4.fill_between([0,1],[0,1], alpha=0.02, color='gray')
ax4.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax4.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
ax4.set_title('Curvas ROC — Todos los Modelos', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8, loc='lower right')
ax4.set_aspect('equal')

# --- Panel 5: Brier Score ---
ax5 = fig.add_subplot(gs[1, 2])
bars = ax5.barh(nombres, df_resultados['Brier Score'], color=colors_bar, edgecolor='white', linewidth=0.8)
ax5.set_xlabel('Brier Score (↓ mejor)', fontsize=11)
ax5.set_title('Calibración (Brier Score)', fontsize=12, fontweight='bold')
for bar, v in zip(bars, df_resultados['Brier Score']):
    ax5.text(v + 0.001, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[0], label='Modelos Lineales'),
    Patch(facecolor=PALETTE[2], label='Árboles'),
    Patch(facecolor=PALETTE[1], label='Boosting'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=11, 
           bbox_to_anchor=(0.5, 0.97), frameon=True)

fig.suptitle('Benchmark Completo — Credit Scoring Tradicional vs Machine Learning\nDataset: German Credit (1,000 obs., 30% default rate)', 
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('../figuras/sesion09_benchmark_completo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Tabla de Resultados Ordenada por AUC:")
display(df_resultados.sort_values('AUC-ROC', ascending=False)
        .assign(**{'AUC-ROC': lambda x: x['AUC-ROC'].round(4),
                   'Gini':    lambda x: (x['Gini']*100).round(2),
                   'KS':      lambda x: (x['KS']*100).round(2),
                   'Brier Score': lambda x: x['Brier Score'].round(4),
                   'CV AUC (5-fold)': lambda x: x['CV AUC (5-fold)'].round(4),
                   'Tiempo (s)': lambda x: x['Tiempo (s)'].round(3)})
        .reset_index(drop=True)
        .style.background_gradient(cmap='RdYlGn', subset=['AUC-ROC', 'Gini', 'KS'])
        .background_gradient(cmap='RdYlGn_r', subset=['Brier Score'])
        .set_caption('Benchmark: Modelos de Credit Scoring')
       )

In [ ]:
# =============================================================================
# ANÁLISIS DE CALIBRACIÓN
# La calibración es crítica en IFRS 9: las PDs deben ser probabilidades reales
# =============================================================================

modelos_calibracion = ['Reg. Logística', 'Random Forest', 'Gradient Boosting']
colores_calib = [PALETTE[0], PALETTE[2], PALETTE[1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (nombre, color) in enumerate(zip(modelos_calibracion, colores_calib)):
    idx = list(modelos_bench.keys()).index(nombre)
    probs = resultados[idx]['probs']
    
    fraction_of_pos, mean_predicted = calibration_curve(y_test, probs, n_bins=10, strategy='quantile')
    
    ax = axes[i]
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6, label='Calibración perfecta')
    ax.plot(mean_predicted, fraction_of_pos, '-o', color=color, lw=2, markersize=8, label=nombre)
    ax.fill_between(mean_predicted, fraction_of_pos, mean_predicted, alpha=0.15, color=color)
    
    brier = resultados[idx]['Brier Score']
    ax.set_title(f'{nombre}\nBrier Score = {brier:.4f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Probabilidad Media Predicha', fontsize=10)
    ax.set_ylabel('Fracción de Positivos Reales', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect('equal')
    
    # Diagrama de calibración
    ax2 = ax.twinx()
    ax2.hist(probs, bins=20, alpha=0.15, color=color, density=True)
    ax2.set_ylabel('Densidad de Scores', fontsize=8, color='gray')
    ax2.tick_params(axis='y', labelcolor='gray', labelsize=7)

fig.suptitle('Diagramas de Calibración — Crítico para IFRS 9 (PD)\nUn modelo bien calibrado sigue la línea diagonal', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figuras/sesion09_calibracion.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Observación clave:")
print("   La Regresión Logística produce probabilidades mejor calibradas de forma nativa.")
print("   Random Forest y Gradient Boosting requieren post-calibración (Platt Scaling, Isotonic Regression).")
print("   Esto es especialmente importante en IFRS 9 donde las PD son insumos regulatorios.")

---

# 🧭 Sección 6 — Framework de Decisión: ¿Cuándo Usar Cada Enfoque?

## 6.1 El Árbol de Decisión del Modelador

No existe un modelo universalmente superior. La decisión depende del contexto. El siguiente framework orientativo está basado en la práctica de la industria bancaria:

```
¿El regulador exige explicabilidad individual de cada decisión?
    ├── SÍ → ¿Tienes suficiente masa crítica de datos (>50k obs.)?
    │         ├── SÍ → ML con herramientas XAI (SHAP) + validación exhaustiva
    │         └── NO → Regresión Logística con WOE (más confiable con muestras pequeñas)
    └── NO → ¿El objetivo es maximizar discriminación (ej. fraud scoring)?
              ├── SÍ → Gradient Boosting (XGBoost/LightGBM) con validación rigurosa
              └── NO → Evaluar caso a caso según datos disponibles
```

## 6.2 Criterios por Tipo de Portafolio

| Tipo de Portafolio | Recomendación | Razón Principal |
|--------------------|---------------|-----------------|
| **Consumo masivo** (tarjetas, personal) | ML (GBM/RF) | Volumen alto, datos alternativos, compite por AUC |
| **PYME / Empresas** | Logística o RF | Menor muestra, variables financieras estructuradas |
| **Hipotecario** | Logística | Regulación estricta, larga duración |
| **Microfinanzas** | RF o GBM | Datos alternativos (telecom, pagos), thin file |
| **IFRS 9 PD** | Logística + calibración | Exige interpretabilidad y trazabilidad regulatoria |
| **Behavioral scoring** | GBM / RF | Datos transaccionales ricos, actualización frecuente |

## 6.3 La Visión Híbrida: Lo Mejor de Ambos Mundos

La tendencia en la banca de vanguardia (2020–2026) es **modelos Champion-Challenger** con enfoque híbrido:

```
Champion Model: Regresión Logística (en producción regulatoria)
    ↕  monitoreo continuo de diferencial de performance
Challenger Model: XGBoost + SHAP (en validación paralela)

Resultado: migración gradual cuando el challenger demuestra:
  1. +3 puntos Gini sostenidos en 6 meses
  2. Calibración equivalente o superior post-ajuste
  3. Explicabilidad aprobada por comité de modelo
  4. Validación independiente por Model Risk Management
```

In [ ]:
# =============================================================================
# VISUALIZACIÓN: Análisis de Overfitting — Train vs Test AUC
# Un ML sin regularización puede sobreajustar mucho más que la RL
# =============================================================================

modelos_overfit = {
    'Reg. Logística':          LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree\n(sin límite)': DecisionTreeClassifier(random_state=42),
    'Decision Tree\n(max_depth=5)': DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42),
    'Random Forest\n(sin límite)': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest\n(regularizado)': RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42, n_jobs=-1),
    'GBM (sin reg.)': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'GBM (regularizado)': GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42),
}

train_aucs, test_aucs, nombres_of = [], [], []

for nombre, modelo in modelos_overfit.items():
    modelo.fit(X_train_proc, y_train)
    train_aucs.append(roc_auc_score(y_train, modelo.predict_proba(X_train_proc)[:,1]))
    test_aucs.append(roc_auc_score(y_test,  modelo.predict_proba(X_test_proc)[:,1]))
    nombres_of.append(nombre)

fig, ax = plt.subplots(figsize=(13, 5))
x_pos = np.arange(len(nombres_of))
width = 0.35

bars_tr = ax.bar(x_pos - width/2, train_aucs, width, label='AUC Train', color=PALETTE[2], alpha=0.7, edgecolor='white')
bars_te = ax.bar(x_pos + width/2, test_aucs,  width, label='AUC Test',  color=PALETTE[1], alpha=0.7, edgecolor='white')

# Flecha de brecha de overfitting
for i, (tr, te) in enumerate(zip(train_aucs, test_aucs)):
    diff = tr - te
    color_diff = 'red' if diff > 0.02 else 'green'
    ax.annotate('', xy=(i + width/2, te + 0.002), xytext=(i - width/2, tr - 0.002),
                arrowprops=dict(arrowstyle='->', color=color_diff, lw=1.5))
    ax.text(i, max(tr, te) + 0.005, f'Δ={diff:.3f}', ha='center', fontsize=7, 
            color=color_diff, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(nombres_of, fontsize=9)
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('Análisis de Overfitting — AUC Train vs AUC Test\nΔ grande en rojo = riesgo de sobreajuste', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.60, 1.05)
ax.axhline(0.70, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../figuras/sesion09_overfitting_analisis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Observación: La regularización es crítica en ML para crédito.")
print("   Un árbol sin restricciones tiene AUC Train ≈ 1.0 pero bajo desempeño real.")
print("   La Regresión Logística es naturalmente parsimoniosa (menor brecha).")

---

# 🏛️ Sección 7 — Gobernanza y Regulación de Modelos de ML

## 7.1 El Marco Regulatorio Global

El uso de Machine Learning en credit scoring está cada vez más regulado. Conocer el marco es **indispensable** para cualquier profesional del sector:

### Regulación Clave

| Marco | Jurisdicción | Impacto en ML Credit Scoring |
|-------|-------------|------------------------------|
| **SR 11-7** | Fed / OCC (USA) | Establece el estándar de Model Risk Management (MRM): documentación, validación independiente, límites de uso |
| **EBA Guidelines on IRB** | Europa | Exige que los modelos PD sean interpretables y auditables |
| **Basilea IV** | Global | Restricciones a modelos internos (IMM); mayor escrutinio de inputs |
| **GDPR / AI Act** | Europa | Derecho a explicación de decisiones automatizadas |
| **Equal Credit Opportunity Act** | USA | Prohíbe discriminación; los modelos ML pueden amplificar sesgos |
| **IFRS 9** | Global | Requiere trazabilidad de PD en provisiones |

## 7.2 SR 11-7: El Estándar de Oro del MRM

La guía SR 11-7 de la Reserva Federal (2011) es el documento más referenciado internacionalmente para la gestión del riesgo de modelos. Define:

**Riesgo de Modelo:** *"El riesgo de consecuencias adversas derivadas de decisiones basadas en resultados de modelos incorrectos o mal usados."*

Para modelos ML, los tres pilares son:

```
┌─────────────────────────────────────────────────────────┐
│               SR 11-7 — Tres Pilares                    │
├────────────────┬────────────────┬────────────────────────┤
│  DESARROLLO    │  VALIDACIÓN    │  GOBERNANZA            │
│                │  INDEPENDIENTE │                        │
│ • Documenta-   │ • Back-testing │ • Políticas de uso     │
│   ción técnica │ • Stress-tests │ • Revisión periódica   │
│ • Justifica-   │ • Benchmarking │ • Inventory de modelos │
│   ción teórica │ • Sensibilidad │ • Gestión del ciclo    │
│ • Supuestos    │               │   de vida              │
└────────────────┴────────────────┴────────────────────────┘
```

## 7.3 El Problema del Sesgo Algorítmico

Los modelos ML pueden **amplificar sesgos históricos** presentes en los datos de entrenamiento. Esto es una preocupación regulatoria creciente:

```python
# Ejemplo conceptual de fuente de sesgo en credit scoring:
# Si históricamente ciertos grupos demográficos tuvieron menos acceso al crédito,
# los datos de entrenamiento estarán sesgados en esa dirección.
# Un modelo ML que aprende de estos datos perpetuará — o ampliará — la inequidad.

# Herramientas de auditoría de fairness (ver Sesión 14):
# - Disparate Impact Ratio
# - Equal Opportunity Difference
# - Fairlearn, Aequitas, IBM AI Fairness 360
```

## 7.4 Buenas Prácticas de Gobernanza ML en Banca

```
1. INVENTARIO DE MODELOS
   Registrar todos los modelos en uso: tipo, uso, owner, última validación

2. DOCUMENTACIÓN TÉCNICA
   Metodología, datos, supuestos, limitaciones, métricas de performance

3. VALIDACIÓN INDEPENDIENTE
   Equipo separado del desarrollador; back-testing en muestra OOT

4. MONITOREO CONTINUO
   PSI (estabilidad de variables), CSI (drift del score), performance mensual

5. PLAN DE CONTINGENCIA
   ¿Qué pasa si el modelo falla? Modelo de respaldo definido y documentado

6. EXPLICABILIDAD
   Para cada decisión adversa: razones principales (SHAP top-3 features)
```

In [ ]:
# =============================================================================
# PREVIEW: Importancia de Variables — Un Vistazo a la Interpretabilidad
# (Profundizaremos en Sesión 14 con SHAP, LIME, PDP)
# =============================================================================

# Importancia nativa de Random Forest
rf_model = modelos_bench['Random Forest']
importancias_rf = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

# Coeficientes de Regresión Logística (como proxy de importancia)
rl_model = modelos_bench['Reg. Logística']
coef_rl = pd.Series(np.abs(rl_model.coef_[0]), index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_n = 10

ax = axes[0]
coef_rl.head(top_n).plot(kind='barh', ax=ax, color=PALETTE[0], edgecolor='white', linewidth=0.8)
ax.invert_yaxis()
ax.set_title('Regresión Logística\n|Coeficiente| (post-estandarización)', fontsize=12, fontweight='bold')
ax.set_xlabel('|Coeficiente|', fontsize=11)
ax.set_ylabel('')

ax = axes[1]
importancias_rf.head(top_n).plot(kind='barh', ax=ax, color=PALETTE[2], edgecolor='white', linewidth=0.8)
ax.invert_yaxis()
ax.set_title('Random Forest\nImportancia de Variables (MDI)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importancia (Gini Impurity)', fontsize=11)
ax.set_ylabel('')

fig.suptitle('Preview: Importancia de Variables — Tradicional vs ML\n(Profundizaremos con SHAP en Sesión 14)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figuras/sesion09_importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Top 5 Variables más Importantes:")
print(f"\n   Reg. Logística: {list(coef_rl.head(5).index)}")
print(f"   Random Forest:  {list(importancias_rf.head(5).index)}")
print()
print("💡 Observación: Ambos modelos identifican variables similares como importantes,")
print("   pero Random Forest puede detectar interacciones no visibles para la RL.")

---

# 🗺️ Sección 8 — Conclusiones y Roadmap del Curso II

## 8.1 Síntesis de la Sesión

### Lo que aprendimos hoy:

| Concepto | Takeaway Clave |
|----------|----------------|
| **Limitaciones de la RL** | No captura no linealidades ni interacciones automáticamente |
| **Valor del ML** | Discriminación superior especialmente con datos ricos y alta dimensionalidad |
| **Trade-off central** | Interpretabilidad ↔ Performance — no hay solución gratuita |
| **Calibración** | La RL es nativa; ML requiere post-calibración (clave para IFRS 9) |
| **Overfitting** | ML sin regularización puede sobreajustar severamente |
| **Regulación** | SR 11-7, EBA y IFRS 9 imponen condiciones al uso de ML en banca |
| **Framework decisión** | El contexto (portafolio, regulación, datos) determina el modelo |

## 8.2 Roadmap del Curso II — Sesiones 9–18

```
SESIÓN 9  ← Estás aquí: Tradicional vs ML (marco conceptual y benchmark)
    ↓
SESIÓN 10: Preparación de Datos para ML
           • Feature engineering específico para crédito
           • Manejo de desbalance de clases (SMOTE, class_weight)
           • Pipeline scikit-learn robusto
    ↓
SESIÓN 11: Árboles de Decisión y Random Forest
           • CART en profundidad; criterios de split
           • Bagging y el poder de la decorrelación
           • Tuning de hiperparámetros
    ↓
SESIÓN 12: Gradient Boosting (XGBoost / LightGBM)
           • Boosting secuencial; función de pérdida logarítmica
           • XGBoost vs LightGBM vs CatBoost
           • Early stopping y regularización
    ↓
SESIÓN 13: Comparación y Selección de Modelos
           • Cross-validation estratificada rigurosa
           • Pruebas estadísticas de diferencia (DeLong)
           • Champion-Challenger framework
    ↓
SESIÓN 14: Interpretabilidad y Explicabilidad
           • SHAP values (TreeSHAP)
           • LIME para instancias individuales
           • PDP y ALE plots
    ↓
SESIÓN 15: IFRS 9 — Introducción y Calibración PD
           • Staging (S1/S2/S3) y SICR
           • PD Point-in-time vs Through-the-cycle
    ↓
SESIÓN 16: Calibración LGD y EAD
           • Modelos de LGD: regresión Beta, Random Forest
           • Modelos de EAD: CCF modeling
    ↓
SESIÓN 17: Modelos de Provisiones IFRS 9
           • ECL = PD × LGD × EAD
           • Forward-looking adjustments
    ↓
SESIÓN 18: Caso Integrador Final
           • Pipeline completo: datos → modelo → score → IFRS 9 → provisiones
```

## 8.3 Recomendaciones para la Próxima Sesión

Antes de la Sesión 10, se recomienda:

1. **Revisar** la documentación de `sklearn.pipeline.Pipeline` y `sklearn.compose.ColumnTransformer`
2. **Leer** el Capítulo 4 de Hands-On Machine Learning (Geron) sobre preprocesamiento
3. **Practicar** la construcción de pipelines con el dataset de la Parte 1
4. **Reflexionar:** ¿En qué portafolio de tu institución crees que ML agregaría más valor? ¿Por qué?

---

## 📚 Referencias Bibliográficas Recomendadas

### Libros Fundamentales

1. **Siddiqi, N. (2017).** *Intelligent Credit Scoring: Building and Implementing Better Credit Risk Scorecards* (2nd ed.). Wiley. *(Capítulo 11: ML en Scoring)*

2. **Géron, A. (2022).** *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3rd ed.). O'Reilly Media. *(Capítulos 1–4: Fundamentos ML)*

3. **Hastie, T., Tibshirani, R., & Friedman, J. (2009).** *The Elements of Statistical Learning* (2nd ed.). Springer. *(Capítulos 9–10: Trees y Boosting)* *(Disponible gratis en: https://hastie.su.domains/ElemStatLearn/)*

4. **James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021).** *An Introduction to Statistical Learning with Applications in Python*. Springer. *(Capítulos 8–9)* *(Disponible gratis en: https://www.statlearning.com/)*

5. **Baesens, B., Roesch, D., & Scheule, H. (2016).** *Credit Risk Analytics: Measurement Techniques, Applications, and Examples in SAS*. Wiley. *(Referencia técnica para scoring en banca)*

### Artículos y Documentos Técnicos

6. **Khandani, A.E., Kim, A.J., & Lo, A.W. (2010).** "Consumer credit-risk models via machine-learning algorithms." *Journal of Banking & Finance, 34*(11), 2767–2787.

7. **Lessmann, S., Baesens, B., Seow, H.V., & Thomas, L.C. (2015).** "Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research." *European Journal of Operational Research, 247*(1), 124–136.

8. **Molnar, C. (2022).** *Interpretable Machine Learning: A Guide for Making Black Box Models Explainable* (2nd ed.). *(Disponible gratis en: https://christophm.github.io/interpretable-ml-book/)*

9. **Federal Reserve (2011).** *SR 11-7: Guidance on Model Risk Management.* Board of Governors of the Federal Reserve System. *(Documento regulatorio fundamental)*

10. **EBA (2020).** *Guidelines on Loan Origination and Monitoring.* European Banking Authority. *(Marco europeo para credit scoring)*

### Recursos Online

11. **Scikit-learn Documentation:** https://scikit-learn.org/stable/supervised_learning.html

12. **XGBoost Documentation:** https://xgboost.readthedocs.io/en/stable/

13. **SHAP Documentation:** https://shap.readthedocs.io/en/latest/

14. **Weights & Biases — Credit Scoring Best Practices:** https://wandb.ai/fully-connected/credit-scoring

---

*Fin de la Sesión 9 — Programa de Especialización en Credit Scoring con Python*  
*Curso II: Credit Scoring y Machine Learning*  
*Docente: Enzo Infantes Zúñiga | enzo.infantes28@gmail.com*

In [ ]:
# =============================================================================
# RESUMEN FINAL DE RESULTADOS DE LA SESIÓN
# =============================================================================

print("=" * 65)
print("  SESIÓN 9 — RESUMEN DE RESULTADOS")
print("  Credit Scoring Tradicional vs Machine Learning")
print("=" * 65)

print("\n📊 BENCHMARK FINAL — Ordenado por AUC-ROC:")
print("-" * 65)
print(f"{'Modelo':<25} {'AUC':>7} {'Gini':>8} {'KS':>8} {'Brier':>8}")
print("-" * 65)

for _, row in df_resultados.sort_values('AUC-ROC', ascending=False).iterrows():
    marker = " ✅" if row['AUC-ROC'] == df_resultados['AUC-ROC'].max() else ""
    print(f"{row['Modelo']:<25} {row['AUC-ROC']:>7.4f} "
          f"{row['Gini']*100:>7.2f}% {row['KS']*100:>7.2f}% "
          f"{row['Brier Score']:>8.4f}{marker}")

print("-" * 65)
best_ml  = df_resultados.loc[df_resultados['Modelo'].str.contains('Gradient|Forest'), 'AUC-ROC'].max()
best_trad = df_resultados.loc[df_resultados['Modelo'].str.contains('Logística'), 'AUC-ROC'].max()
delta    = best_ml - best_trad

print(f"\n🔑 Mejor modelo ML     : AUC = {best_ml:.4f}")
print(f"   Reg. Logística      : AUC = {best_trad:.4f}")
print(f"   Ganancia ML vs RL   : +{delta*100:.2f} puntos AUC")
print(f"   Ganancia Gini       : +{delta*200:.2f} puntos Gini")

print("\n💡 CONCLUSIONES CLAVE:")
conclusiones = [
    "1. ML supera a la Regresión Logística en discriminación, pero el margen es contextual.",
    "2. La Regresión Logística sigue siendo el estándar regulatorio y de interpretabilidad.",
    "3. Los modelos ML requieren regularización para evitar overfitting severo.",
    "4. La calibración de probabilidades es crítica para IFRS 9 — RL gana nativamente.",
    "5. El framework Champion-Challenger permite transición segura hacia ML.",
]
for c in conclusiones:
    print(f"   {c}")

print("\n📅 PRÓXIMA SESIÓN: Preparación de Datos para Modelos de Machine Learning")
print("   • Feature Engineering específico para crédito")
print("   • Manejo de desbalance de clases (SMOTE, pesos)")
print("   • Construcción de pipelines robustos con scikit-learn")
print("=" * 65)